<a href="https://colab.research.google.com/github/jessica-aaao/ChillPlace/blob/master/ChordExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [1]:
!python3 -m pip install -q -U "yt-dlp[default]"
!pip install -q -U openai-whisper
!pip install -q -U demucs

In [2]:
import requests
import json
import pandas as pd
import os
import re
import unicodedata

In [3]:
from google.colab import drive
from IPython.display import display
from bs4 import BeautifulSoup

In [4]:
!git clone  https://github.com/mikezzb/lyrics-sync.git
!git clone https://github.com/filipecalegario/ISMIR2019-Large-Vocabulary-Chord-Recognition.git

fatal: destination path 'lyrics-sync' already exists and is not an empty directory.
fatal: destination path 'ISMIR2019-Large-Vocabulary-Chord-Recognition' already exists and is not an empty directory.


In [5]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Common

In [6]:
class SongUrls:
    def __init__(self, name, audio, lyrics, chords):
        self.name = name
        self.audio = audio
        self.lyrics = lyrics
        self.chords = chords

    def get_name(self):
        return self.name

    def get_audio_url(self):
        return self.audio

    def get_lyrics_url(self):
        return self.lyrics

    def get_chords_url(self):
        return self.chords

In [7]:
def get_urls():
    print(f'Fetching urls...\n\n')

    file_path = '/content/drive/My Drive/TCC/CodeData/songs.csv'
    songs = pd.read_csv(file_path)

    print(f'Urls fetched!\n\n')

    return songs

In [8]:
def get_songs_from_csv():
    songs = get_urls()

    song_urls = []

    print(f'Creating SongUrls...\n\n')

    for index, row in songs.iterrows():
        song_name = slugify(row['Song Name'])
        audio_url = row['Audio URL']
        lyrics_url = row['Lyrics URL']
        chords_url = row['Chords URL']

        song_urls.append(SongUrls(song_name, audio_url, lyrics_url, chords_url))

    print(f'SongUrls created!\n\n')

    return song_urls


In [9]:
def slugify(raw_song_name):
    song_name = raw_song_name.lower()

    song_name = unicodedata.normalize('NFKD', song_name)
    song_name = song_name.encode('ascii', 'ignore').decode('ascii')

    song_name = re.sub(r'[^a-z0-9]+', '_', song_name)

    song_name = song_name.strip('_')

    return song_name

# Sound

In [10]:
def extract_voice_from_audio(audio_path, song_name):
    print(f'Extracting vocals from {audio_path}...\n\n')

    !demucs --two-stems=vocals {audio_path} -o vocals/

    return f"/content/vocals/htdemucs/{song_name}/vocals.wav"


In [11]:
def extract_audio_from_video(youtube_url, song_name):
    cookies_path = '/content/cookies.txt'
    output_path = f"/content/audios/{song_name}.wav"

    !yt-dlp {youtube_url} --audio-format "wav" --cookies {cookies_path} -x -o {output_path}  -q

    return f"{output_path}"

In [12]:
def extract_sound_recording(youtube_url, song_name):
    output_path = extract_audio_from_video(youtube_url, song_name)

    print(f'Audio saved as {output_path}!\n\n')

    return output_path

# Chords

## Chords Extraction from Audio

In [31]:
def extract_chords_from(song_path, song_name):
    """Extrai a cifra, com o timestamp, a partir da música"""
    print(f"Extracting chords from {song_path}...\n\n")

    output_folder = "/content/chords"
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    output_path = f"{output_folder}/{song_name}.lab"

    %cd ISMIR2019-Large-Vocabulary-Chord-Recognition
    !pip install -q -r requirements.txt
    !python chord_recognition.py {song_path} {output_path}
    %cd ..
    return output_path

## Chords Sync to Lyrics

### Chords Parsing

In [51]:
def simplify_chord(chord):
    chord = chord.replace(":", "")
    chord = chord.replace("min", "m")
    chord = chord.replace("maj", "")
    chord = chord.replace("hdim7", "m7(b5)")
    chord = chord.replace("hdim", "m7(b5)")
    chord = chord.replace("sus4(b7)", "7sus4")

    return chord

In [50]:
def clean_chords(chords):
    """
    Dada uma lista de acordes (ordenados por tempo),
    remove repetições imediatas do mesmo acorde.
    """
    if not chords:
        return []
    result = [chords[0]]

    for i in range(1, len(chords)):
        current = chords[i]
        previous = result[-1]
        if current["chord"] != previous["chord"]:
            result.append(current)
        else:
            previous["end"] = current["end"]
    return result

In [49]:
def parse_chords(chords_path):
    # Ler ACORDES do arquivo LAB
    chords = []
    with open(chords_path, 'r', encoding='utf-8') as file:
        for line in file:
            start, end, chord = line.strip().split('\t')
            chord = simplify_chord(chord)
            chords.append({
                "start": float(start),
                "end": float(end),
                "chord": chord
            })
        return clean_chords(chords)

### Lyrics Parsing

In [62]:
def parse_lyrics(timestamped_lyrics_path, lyrics_path):
    # Ler LETRA_TRANSCRITA do arquivo CSV
    timestamped_lyrics = pd.read_csv(timestamped_lyrics_path)
    timestamped_per_line = []

    with open(lyrics_path, 'r', encoding='utf-8') as file:
        for index, line in enumerate(file):
            line = line.strip().split()
            words_in_line = len(line)

            timestamped_words = timestamped_lyrics.iloc[:words_in_line]
            # Remove as palavras já processadas do DataFrame
            timestamped_lyrics = timestamped_lyrics.iloc[words_in_line:]
            # Adiciona as palavras processadas à lista
            timestamped_per_line.append([timestamped_words])

    return timestamped_per_line

### General

In [42]:
def format_song_with_chords(timestamped_lyrics_path, chords_path):
    """
    Formata a LETRA_TRANSCRITA com acordes no formato tradicional de cifras.
    """
    lyrics = parse_lyrics(timestamped_lyrics_path)
    chords = parse_chords(chords_path)

    lyrics_with_chords = overlay_chords_on_transcribed(timestamped_lyrics, chords)



    # Formatar saída com acordes e palavras
    formatted_output = format_transcribed_with_chords(lyrics_with_chords)
    print(formatted_output)
    return formatted_output

In [43]:
def overlay_chords_on_transcribed(timestamped_lyrics, chords):
    """
    Associa acordes às palavras da LETRA_TRANSCRITA com base nos tempos de ACORDES.
    """
    result = []
    for index, row in timestamped_lyrics.iterrows():
        start = row["start"]
        end = row["end"]
        word = row["label"]
        chord = None
        chord_start = None

        # Encontrar o acorde correspondente
        if start is not None and end is not None:
            for chord_start, chord_end, chord_name in chords:
                if chord_start <= start < chord_end:
                    chord = chord_name
                    chord_start = start
                    break

        result.append({
            "word": word,
            "start": start,
            "end": end,
            "chord": chord,
            "chord_start": chord_start
        })

    return result

In [44]:
def align_chord_over_word(word_info):
    word = word_info["word"]
    word_start = word_info["start"]
    word_end = word_info["end"]
    chord = word_info["chord"]
    chord_start = word_info["chord_start"]

    word_duration = word_end - word_start
    ratio = (chord_start - word_start) / word_duration

    word_len = len(word)
    word_index = int(round(ratio * (word_len - 1)))
    word_index = max(0, min(word_index, word_len - 1))

    left_padding = " " * word_index
    chord_with_padding = f"{left_padding}{chord}"

    return chord_with_padding

In [45]:
def format_transcribed_with_chords(lyrics_with_chords):
    """
    Gera o formato de saída com acordes acima das palavras.
    """
    formatted_output = []
    chord_line = []
    lyrics_line = []

    for word_info in lyrics_with_chords:
        word = word_info["word"]
        chord = word_info["chord"]

        if chord:
            chord_line.append(align_chord_over_word(word_info))
        else:
            chord_line.append(" " * len(word))

        lyrics_line.append(word)

        # Adiciona quebra de linha para segmentos
        if word.endswith((".", ",", "?")):
            formatted_output.append(" ".join(chord_line).rstrip())
            formatted_output.append(" ".join(lyrics_line).rstrip())
            formatted_output.append("")
            chord_line = []
            lyrics_line = []

    # Adiciona o restante, se existir
    if chord_line or lyrics_line:
        formatted_output.append(" ".join(chord_line).rstrip())
        formatted_output.append(" ".join(lyrics_line).rstrip())

    return "\n".join(formatted_output)

In [46]:
def simplify_chords_with_lyrics(chords_line, lyrics_line):
    """
    Simplifica uma linha de acordes e palavras associadas.
    Remove redundâncias e organiza a saída.
    """
    chords = chords_line.split()
    lyrics = lyrics_line.split()

    simplified_chords = []
    simplified_lyrics = []

    # Último acorde usado
    last_chord = None

    for chord, word in zip(chords, lyrics):
        if chord != last_chord:
            simplified_chords.append(chord)
            simplified_lyrics.append(word)
            last_chord = chord
        else:
            simplified_chords.append("")  # Espaço vazio na linha de acordes
            simplified_lyrics.append(word)

    # Retorna linhas formatadas
    return " ".join(simplified_chords).rstrip(), " ".join(simplified_lyrics).rstrip()

In [47]:
def format_clean_chords_and_lyrics(raw_chords_lyrics):
    """
    Formata o texto contendo acordes redundantes e letras em um formato limpo e tradicional.
    """
    formatted_output = []
    lines = raw_chords_lyrics.strip().split("\n")

    # Processa par de linhas (chord, lyrics)
    for i in range(0, len(lines), 2):
        if i + 1 < len(lines):  # Certifique-se de que há uma linha de letras
            chords_line = lines[i]
            lyrics_line = lines[i + 1]

            # Simplificar a linha
            simplified_chords, simplified_lyrics = simplify_chords_with_lyrics(chords_line, lyrics_line)

            # Adicionar ao resultado
            formatted_output.append(simplified_chords)
            formatted_output.append(simplified_lyrics)
            formatted_output.append("")  # Linha em branco para separar blocos

    return "\n".join(formatted_output).strip()



---



# Lyrics

## Lyrics Extraction from Webpage

In [20]:
def save_to_file(data, song_name):
    folder_path = "/content/lyrics"
    path = f"{folder_path}/{song_name}.txt"

    if not os.path.exists(folder_path):
        os.makedirs(folder_path)

    with open(path, 'w') as file:
        file.write(data)

    print(f'Saved as {path}')

    return path

In [21]:
def extract_lyrics_from_html(html):
    """Extrai a letra da página HTML fornecida"""

    print(f'Fetching lyrics...!\n\n')

    lyricsTag = html.find('div', class_='lyric-original')
    lyrics = ""

    for p in lyricsTag.find_all('p'):
        for br in p.find_all('br'):
            br.replace_with('\n')
        lyrics += p.get_text() + "\n"

    print(f'Lyrics fetched!\n\n')

    return lyrics


In [22]:
def get_lyrics_from_webpage(lyric_url, song_name):
    """Obtém a página web e extrai a letra"""

    print(f'Fetching webpage {lyric_url}...\n\n')
    response = requests.get(lyric_url)

    if response.status_code == 200:
        print(f'Webpage fetched!\n\n')
        htmlContent = BeautifulSoup(response.content, 'html.parser')

        lyrics = extract_lyrics_from_html(htmlContent)
        lyrics_path = save_to_file(lyrics, song_name)

        return lyrics_path
    else:
        print(f"Failed to fetch the webpage. Status code: {response.status_code}\n\n")

## Lyrics Sync to Audio

### Transcription + Manual Adjustment Attempt

In [60]:
def adjust_lyrics(transcribed_segments, comparison_lyrics):
    """Compara a letra transcrita e a letra extraída da Web, corrigindo a letra transcrita"""

    print(f"Adjusting lyrics...\n\n")

    for index, segment in enumerate(transcribed_segments):
        try:
            line = comparison_lyrics[index]

            if segment['text'].lower() != line.lower():
                segment['text'] = line

            words = line.split()
            # Testar com músicas não "especiais", mais enxutas
            if len(words) != len(segment['words']):
                print(f"Word count mismatch: {len(words)} != {len(segment['words'])}")
                continue

            for i, word in enumerate(words):
                try:
                    transcribed_word = segment['words'][i]['word']

                    if transcribed_word.lower().strip() != word.lower():
                        print(f"Word mismatch: {transcribed_word} != {word}")

                        segment['words'][i]['word'] = word
                except:
                    print(f"Word not found: {word}")
                    continue
        except:
            print(f"Line not found: {segment['text']}")
            continue

    print(f"Lyrics adjusted:\n{json.dumps(transcribed_lyrics, indent=2)}\n\n")

    return transcribed_lyrics

In [61]:
def transcribe_audio(audio_path):
    """Transcreve o áudio usando o modelo Whisper"""

    print(f"Transcribing audio...\n\n")

    model = whisper.load_model("turbo", device='cuda')
    result = model.transcribe(audio_path, word_timestamps=True)

    print(f"Audio Transcribed!\n\n")

    return result['segments']


### Using Lyrics-Sync

In [23]:
def create_output_folder():
    output_folder = "/content/lyrics-sync/output"
    vocals_folder = output_folder + "/vocals"
    words_folder = output_folder + "/words"
    lrc_folder = output_folder + "/lrc"

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    if not os.path.exists(vocals_folder):
        os.makedirs(vocals_folder)

    if not os.path.exists(words_folder):
        os.makedirs(words_folder)

    if not os.path.exists(lrc_folder):
        os.makedirs(lrc_folder)

In [24]:
def get_timestamps(audio_path, lyrics_path, song_name):
    print("Installing conda...")
    !wget -c https://repo.continuum.io/archive/Anaconda3-2024.10-1-Linux-x86_64.sh
    !chmod +x Anaconda3-2024.10-1-Linux-x86_64.sh
    !bash ./Anaconda3-2024.10-1-Linux-x86_64.sh -b -f -p /usr/local
    print("Conda installed!")

    print("Installing lsync...")
    %cd /content/lyrics-sync/
    !conda env update -f environment.yml
    !source activate lsync

    print("Lsync installed!")
    from lsync import LyricsSync

    print("Extracting timestamps...")
    lsync = LyricsSync()
    words, lrc = lsync.sync(audio_path, lyrics_path)
    print("Timestamps extracted!")

    return f"/content/lyrics-sync/output/words/{song_name}.csv"
    %cd ..

In [25]:
def get_synced_lyrics(lyric_url, audio_path, song_name):
    """Obtém a letra sincronizada com o áudio e corrigida"""

    print(f"Getting synced lyrics...\n\n")

    lyrics_path = get_lyrics_from_webpage(lyric_url, song_name)
    timestamps_path = get_timestamps(audio_path, lyrics_path, song_name)

    print(f"Synced lyrics ready!\n\n")

    return timestamps_path

# MAIN

In [26]:
# Extrai dados do csv e converte pra [SongUrls]
songs = get_songs_from_csv()

song = songs[2]
song_name = song.get_name().replace(" ", "_")
print(f"Generating \'{song_name}\' chord sheet...")

Fetching urls...


Urls fetched!


Creating SongUrls...


SongUrls created!


Generating 'petrolina_juazeiro' chord sheet...


In [29]:
# Baixa músicas do youtube
song_path = extract_sound_recording(song.get_audio_url(), song_name)

ERROR: unable to download video data: HTTP Error 403: Forbidden
Audio saved as /content/audios/petrolina_juazeiro.wav!




In [33]:
chords_path = extract_chords_from(song_path, song_name)

Extracting chords from /content/audios/petrolina_juazeiro.wav...


/content/ISMIR2019-Large-Vocabulary-Chord-Recognition
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 36.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 4.1 MB/s eta 0:00:00
/content/ISMIR2019-Large-Vocabulary-Chord-Recognition/mir/nn/train.py:68: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See

In [34]:
# Extrai letras com timestamp
create_output_folder()
lyrics_timestamped_path = get_synced_lyrics(song.get_lyrics_url(), song_path, song_name)

Getting synced lyrics...


Fetching webpage https://www.letras.mus.br/alceu-valenca/400650/...


Webpage fetched!


Fetching lyrics...!


Lyrics fetched!


Saved as /content/lyrics/petrolina_juazeiro.txt
Installing conda...
--2025-03-14 00:43:04--  https://repo.continuum.io/archive/Anaconda3-2024.10-1-Linux-x86_64.sh
Resolving repo.continuum.io (repo.continuum.io)... 104.18.177.84, 104.18.176.84, 2606:4700::6812:b154, ...
Connecting to repo.continuum.io (repo.continuum.io)|104.18.177.84|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh [following]
--2025-03-14 00:43:04--  https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.191.158, 104.16.32.241, 2606:4700::6810:bf9e, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.191.158|:443... connected.
HTTP request sent, awaiting response... 200 OK

Downloading: "https://dl.fbaipublicfiles.com/demucs/hybrid_transformer/955717e8-8726e21a.th" to /root/.cache/torch/hub/checkpoints/955717e8-8726e21a.th


Extracting timestamps...


100%|██████████| 80.2M/80.2M [00:00<00:00, 102MB/s]
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-960h-lv60-self and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Timestamps extracted!
Synced lyrics ready!




In [20]:
raw_chords = format_song_with_chords("/content/petrolina_juazeiro.csv", "/content/petrolina_juazeiro.lab")

Am Am Am Am Am
Na margem do São Francisco,

E E E Am Am Am E E Am Am Dm Dm Dm Dm Am Am Am Am E
nasceu a beleza E a natureza ela conservou Jesus abençoou com sua mão divina Pra não morrer de saudade,

E E E E Am Am Dm Dm Dm Dm Am Am Am Am Am
vou voltar pra Petrolina (Jesus abençoou com sua mão divina) (Pra não morrer de saudade,

E E E E Am Am Am E E
vou voltar pra Petrolina) Do outro lado do rio,

E E E Am Am Am Am E E E E Am Am Dm
tem uma cidade Que na minha mocidade eu visitava todo dia Atravessava a ponte,

Dm
ai,

Dm Dm Am Am Am
que alegria! Chegava em Juazeiro,

Dm E E Am Am Am
Juazeiro da Bahia (Atravessava a ponte,

Dm
ai,

Dm Dm Am Am Am
que alegria!) (Chegava em Juazeiro,

Dm E E Am Am Am Dm Dm Dm Dm Am Am Am Am Am E:7 E:7 E:7 E:7 E:7 Am Am Am Dm Dm Dm Dm Am Am Am Am E:7 E:7 E:7 E:7 E:7 Am
Juazeiro da Bahia) Hoje me lembro que no tempo de criança Esquisito era a carranca e o apito do trem Mas achava lindo quando a ponte levantava E o vapor passava num gostoso vai e vem Petroli

In [15]:
final_chords = format_clean_chords_and_lyrics(raw_chords)

In [63]:
parse_lyrics(lyrics_timestamped_path, "/content/lyrics/petrolina_juazeiro.txt")

[[        label  start    end
  0          Na  16.54  16.76
  1      margem  16.78  17.24
  2          do  17.28  17.40
  3         São  17.42  17.62
  4  Francisco,  17.64  18.26
  5      nasceu  18.30  18.70
  6           a  18.80  18.96
  7      beleza  18.98  20.42],
 [        label  start    end
  8           E  20.56  20.72
  9           a  20.80  20.90
  10   natureza  20.92  21.78
  11        ela  21.86  22.14
  12  conservou  22.18  23.72],
 [       label  start    end
  13     Jesus  23.76  24.22
  14  abençoou  24.28  25.06
  15       com  25.08  25.24
  16       sua  25.28  25.68
  17       mão  25.70  25.92
  18    divina  25.94  26.90],
 [        label  start    end
  19        Pra  26.92  27.12
  20        não  27.16  27.34
  21     morrer  27.38  27.80
  22         de  27.82  28.00
  23   saudade,  28.04  28.66
  24        vou  28.70  28.86
  25     voltar  28.88  29.28
  26        pra  29.30  29.50
  27  Petrolina  29.52  30.62],
 [       label  start    end
  28    (J